# Experiment 5: Subword Tokenization, Part-of-Speech Tagging, and Word2Vec Embeddings

This notebook demonstrates subword tokenization (BPE and SentencePiece, both pretrained and from scratch), Part-of-Speech (POS) tagging (sentence-level using spaCy and NLTK, and corpus-level frequency profiling), and Word2Vec distributed word representations (Continuous Bag of Words, Skip-Gram, and Cosine Similarity).

## 1. Setup and Environment Imports

Import required standard libraries, data manipulation tools, subword tokenizer frameworks, NLP packages, and Gensim Word2Vec.

In [1]:
import string
import pandas as pd
from collections import Counter, defaultdict
import spacy
import nltk
import tiktoken
from transformers import AutoTokenizer
import sentencepiece as spm
from gensim.models import Word2Vec

nlp = spacy.load('en_core_web_sm')
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


True

## Experiment 5.1: Subword-Level Tokenization Using BPE (Byte Pair Encoding)

Tokenize text into subwords and display tokens and token IDs: (a) Using pretrained GPT-2 BPE model, and (b) Without pretrained model (BPE algorithm from scratch trained on `input_sub_word_data.txt`).

In [2]:
with open('input_sub_word_data.txt', 'r', encoding='utf-8') as f:
    text_5 = f.read()

sample_sentence = "Natural language processing is a branch of artificial intelligence that enables computers to understand, interpret, and generate human language."

enc = tiktoken.get_encoding("gpt2")
token_ids_5_1_a = enc.encode(sample_sentence)
tokens_5_1_a = [enc.decode([tid]) for tid in token_ids_5_1_a]

df_5_1_a = pd.DataFrame({'Token': tokens_5_1_a, 'Token ID': token_ids_5_1_a})
print("=== 5.1.a: Pretrained BPE Tokenization (GPT-2) ===")
print(df_5_1_a.head(10))

words = text_5.lower().translate(str.maketrans('', '', string.punctuation)).split()
vocab = Counter([' '.join(list(w)) + ' </w>' for w in words])

def get_stats(v):
    pairs = defaultdict(int)
    for word, freq in v.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, v):
    v_out = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in v:
        v_out[word.replace(bigram, replacement)] = v[word]
    return v_out

num_merges = 30
merges = []
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    vocab = merge_vocab(best_pair, vocab)
    merges.append(best_pair)

subwords = set()
for word in vocab:
    for sym in word.split():
        subwords.add(sym)
token_to_id = {sym: idx for idx, sym in enumerate(sorted(list(subwords)), 1)}

def tokenize_word(w, learned_merges):
    symbols = list(w) + ['</w>']
    for pair in learned_merges:
        i = 0
        while i < len(symbols) - 1:
            if symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                symbols = symbols[:i] + [pair[0] + pair[1]] + symbols[i+2:]
            else:
                i += 1
    return symbols

clean_words = sample_sentence.lower().translate(str.maketrans('', '', string.punctuation)).split()
scratch_tokens = []
for w in clean_words:
    scratch_tokens.extend(tokenize_word(w, merges))

scratch_token_ids = [token_to_id.get(tok, -1) for tok in scratch_tokens]
df_5_1_b = pd.DataFrame({'Token': scratch_tokens, 'Token ID': scratch_token_ids})

print("\n=== 5.1.b: BPE From Scratch (Trained on input_sub_word_data.txt) ===")
print(df_5_1_b.head(10))

=== 5.1.a: Pretrained BPE Tokenization (GPT-2) ===
           Token  Token ID
0        Natural     35364
1       language      3303
2     processing      7587
3             is       318
4              a       257
5         branch      8478
6             of       286
7     artificial     11666
8   intelligence      4430
9           that       326

=== 5.1.b: BPE From Scratch (Trained on input_sub_word_data.txt) ===
  Token  Token ID
0     n        42
1    at         8
2     u        63
3     r        49
4    al         4
5  </w>         1
6     l        35
7    an         5
8     g        26
9     u        63


## Experiment 5.2: Subword-Level Tokenization Using SentencePiece Tokenizer

Tokenize text into subwords and display tokens and token IDs: (a) Using pretrained SentencePiece model (T5-Small), and (b) Without pretrained model (training custom SentencePiece BPE model on `input_sub_word_data.txt`).

In [3]:
t5_tokenizer = AutoTokenizer.from_pretrained('t5-small')
t5_token_ids = t5_tokenizer.encode(sample_sentence)
t5_tokens = t5_tokenizer.convert_ids_to_tokens(t5_token_ids)

df_5_2_a = pd.DataFrame({'Token': t5_tokens, 'Token ID': t5_token_ids})
print("=== 5.2.a: Pretrained SentencePiece Tokenization (T5-Small) ===")
print(df_5_2_a.head(10))

spm.SentencePieceTrainer.train(
    input='input_sub_word_data.txt',
    model_prefix='spm_custom',
    vocab_size=300,
    model_type='bpe',
    character_coverage=1.0
)

sp_processor = spm.SentencePieceProcessor(model_file='spm_custom.model')
sp_tokens = sp_processor.encode_as_pieces(sample_sentence)
sp_token_ids = sp_processor.encode_as_ids(sample_sentence)

df_5_2_b = pd.DataFrame({'Token': sp_tokens, 'Token ID': sp_token_ids})
print("\n=== 5.2.b: SentencePiece From Scratch (Trained on input_sub_word_data.txt) ===")
print(df_5_2_b.head(10))

=== 5.2.a: Pretrained SentencePiece Tokenization (T5-Small) ===
           Token  Token ID
0       ▁Natural      6869
1      ▁language      1612
2    ▁processing      3026
3            ▁is        19
4              ▁         3
5              a         9
6        ▁branch      6421
7            ▁of        13
8    ▁artificial      7353
9  ▁intelligence      6123

=== 5.2.b: SentencePiece From Scratch (Trained on input_sub_word_data.txt) ===
         Token  Token ID
0           ▁N       213
1       atural       174
2    ▁language       118
3  ▁processing       178
4          ▁is        84
5           ▁a         7
6           ▁b        32
7            r       259
8           an        15
9           ch       210


## Experiment 5.3: Part-of-Speech (POS) Tagging on a Given Sentence

Apply POS tagging to "The young student is reading an interesting book in the library." and display unique tokens, POS tags, and tag descriptions: (a) Using spaCy, and (b) Using NLTK.

In [4]:
sentence_5_3 = "The young student is reading an interesting book in the library."

doc_5_3 = nlp(sentence_5_3)
spacy_results = [
    {'Unique Token': token.text, 'POS Tag': token.pos_, 'Description': spacy.explain(token.pos_)}
    for token in doc_5_3 if not token.is_punct
]
df_5_3_spacy = pd.DataFrame(spacy_results).drop_duplicates(subset=['Unique Token'])
print("=== 5.3.a: POS Tagging Using spaCy ===")
print(df_5_3_spacy)

tokens_nltk = nltk.word_tokenize(sentence_5_3)
nltk_tags = nltk.pos_tag(tokens_nltk)

nltk_tag_descriptions = {
    'DT': 'determiner',
    'JJ': 'adjective',
    'NN': 'noun, singular',
    'VBZ': 'verb, 3rd person singular present',
    'VBG': 'verb, gerund or present participle',
    'IN': 'preposition or conjunction, subordinating',
    '.': 'punctuation'
}

nltk_results = [
    {'Unique Token': word, 'POS Tag': tag, 'Description': nltk_tag_descriptions.get(tag, tag)}
    for word, tag in nltk_tags if tag != '.'
]
df_5_3_nltk = pd.DataFrame(nltk_results).drop_duplicates(subset=['Unique Token'])
print("\n=== 5.3.b: POS Tagging Using NLTK ===")
print(df_5_3_nltk)

=== 5.3.a: POS Tagging Using spaCy ===
   Unique Token POS Tag Description
0           The     DET  determiner
1         young     ADJ   adjective
2       student    NOUN        noun
3            is     AUX   auxiliary
4       reading    VERB        verb
5            an     DET  determiner
6   interesting     ADJ   adjective
7          book    NOUN        noun
8            in     ADP  adposition
9           the     DET  determiner
10      library    NOUN        noun



=== 5.3.b: POS Tagging Using NLTK ===
   Unique Token POS Tag                                Description
0           The      DT                                 determiner
1         young      JJ                                  adjective
2       student      NN                             noun, singular
3            is     VBZ          verb, 3rd person singular present
4       reading     VBG         verb, gerund or present participle
5            an      DT                                 determiner
6   interesting      JJ                                  adjective
7          book      NN                             noun, singular
8            in      IN  preposition or conjunction, subordinating
9           the      DT                                 determiner
10      library      NN                             noun, singular


## Experiment 5.4: Corpus-Level POS Tagging and Frequency Profiling

Perform POS tagging on `input_sub_word_data.txt` using spaCy and display unique tokens, POS tags, descriptions, and their occurrence frequencies.

In [5]:
doc_5_4 = nlp(text_5)

token_pos_pairs = [
    (token.text.lower(), token.pos_, spacy.explain(token.pos_))
    for token in doc_5_4 if token.is_alpha
]

pair_counts = Counter(token_pos_pairs)

df_5_4 = pd.DataFrame([
    {'Unique Token': token, 'POS Tag': pos, 'Description': desc, 'Frequency': freq}
    for (token, pos, desc), freq in pair_counts.most_common()
])

print("=== 5.4: Corpus-Level POS Tagging & Frequency (Top 15 Most Frequent Pairs) ===")
print(df_5_4.head(15))

=== 5.4: Corpus-Level POS Tagging & Frequency (Top 15 Most Frequent Pairs) ===
   Unique Token POS Tag               Description  Frequency
0           the     DET                determiner         47
1             a     DET                determiner         28
2           and   CCONJ  coordinating conjunction         27
3           can     AUX                 auxiliary         23
4            of     ADP                adposition         22
5            be     AUX                 auxiliary         15
6            is     AUX                 auxiliary         14
7            as     ADP                adposition         13
8           bpe   PROPN               proper noun         13
9      language    NOUN                      noun         12
10         word    NOUN                      noun         12
11    tokenizer    NOUN                      noun         11
12        units    NOUN                      noun         11
13        words    NOUN                      noun         11
14    

## Experiment 5.5: Word2Vec Continuous Bag-of-Words (CBOW) Model

Train a CBOW Word2Vec model on `corpus.txt`, display the vector representation of a selected word, and identify the three most similar words.

In [6]:
with open('corpus.txt', 'r', encoding='utf-8') as f:
    corpus_sentences = [line.strip().lower().split() for line in f if line.strip()]

cbow_model = Word2Vec(corpus_sentences, vector_size=50, window=3, min_count=1, sg=0, epochs=100, seed=42)

selected_word = 'learning'
cbow_vector = cbow_model.wv[selected_word]

print("=== 5.5: Word2Vec CBOW Model ===")
print(f"Selected Word: '{selected_word}'")
print(f"Vector Representation (Dimension: {len(cbow_vector)}):")
print(cbow_vector)

print(f"\nThree Most Similar Words to '{selected_word}':")
similar_words_cbow = cbow_model.wv.most_similar(selected_word, topn=3)
for word, score in similar_words_cbow:
    print(f"  {word:15} -> Cosine Similarity: {score:.4f}")

=== 5.5: Word2Vec CBOW Model ===
Selected Word: 'learning'
Vector Representation (Dimension: 50):
[-0.01545862  0.01201779  0.00628504 -0.00188551 -0.00196618  0.0155963
 -0.01905005  0.00888532 -0.01197546 -0.01723391  0.00338268  0.02136771
  0.00842204  0.01136455  0.00922062  0.01394025  0.00094621 -0.01672747
  0.01348536 -0.00320153 -0.00104447 -0.00767614 -0.01315929  0.01761156
  0.01475049  0.00452069 -0.00411342  0.01137589  0.00084198 -0.00261616
 -0.00147649 -0.00988968 -0.01794058  0.00376873  0.01560375 -0.01618983
  0.01424317  0.01125406 -0.01080052  0.00712037 -0.0115352   0.01166986
  0.01026188 -0.00520472 -0.01512822  0.01772667 -0.00103597  0.01646561
  0.00779918  0.00944646]

Three Most Similar Words to 'learning':
  uses            -> Cosine Similarity: 0.2450
  changing        -> Cosine Similarity: 0.1526
  useful          -> Cosine Similarity: 0.1390


## Experiment 5.6: Word2Vec Skip-Gram Model

Train a Skip-Gram Word2Vec model on `corpus.txt`, retrieve the vector for "language", and display the five most similar words.

In [7]:
sg_model = Word2Vec(corpus_sentences, vector_size=50, window=3, min_count=1, sg=1, epochs=100, seed=42)

target_word = 'language'
sg_vector = sg_model.wv[target_word]

print("=== 5.6: Word2Vec Skip-Gram Model ===")
print(f"Target Word: '{target_word}'")
print(f"Vector Representation (Dimension: {len(sg_vector)}):")
print(sg_vector)

print(f"\nFive Most Similar Words to '{target_word}':")
similar_words_sg = sg_model.wv.most_similar(target_word, topn=5)
for word, score in similar_words_sg:
    print(f"  {word:15} -> Cosine Similarity: {score:.4f}")

=== 5.6: Word2Vec Skip-Gram Model ===
Target Word: 'language'
Vector Representation (Dimension: 50):
[-5.0178464e-03  1.6547572e-02 -9.8595716e-05  8.3863884e-03
 -1.5498170e-03 -8.6833732e-03  9.7164633e-03  1.8958339e-02
 -9.5581328e-03  1.0816011e-02 -8.7142205e-03  9.5605999e-03
  1.1073690e-02 -1.5550981e-03  9.7048329e-03 -8.2582412e-03
 -1.6588898e-02 -1.6872775e-02 -2.2450807e-03  1.5475531e-02
 -1.5193381e-02 -2.7066786e-03  8.1174104e-03 -1.1617492e-02
  1.0004415e-02 -8.1902724e-03  1.2274580e-02  2.8421176e-03
  1.4466500e-03 -1.2970173e-02 -1.2414800e-03  1.4640452e-02
 -1.9627124e-02  1.0781647e-02 -2.2858515e-04  9.2316763e-03
  6.4206934e-03 -3.2627122e-03 -8.4860427e-03  5.5090999e-03
 -1.3726818e-02  3.7474441e-03 -1.5350362e-02  6.0658604e-03
  5.8671203e-03 -1.7002091e-02  1.0749331e-02 -3.1626029e-03
  1.1628790e-02 -1.8882850e-02]

Five Most Similar Words to 'language':
  uses            -> Cosine Similarity: 0.2892
  love            -> Cosine Similarity: 0.1968
 

## Experiment 5.7: Cosine Similarity Between Word Pairs Using Word2Vec

Calculate the cosine similarity between selected word pairs ("machine" & "learning", "natural" & "language", "deep" & "learning") using the trained Word2Vec model.

In [8]:
word_pairs = [
    ('machine', 'learning'),
    ('natural', 'language'),
    ('deep', 'learning')
]

print("=== 5.7: Cosine Similarity Between Word Pairs ===")
similarity_records = []
for w1, w2 in word_pairs:
    sim_score = sg_model.wv.similarity(w1, w2)
    similarity_records.append({'Word 1': w1, 'Word 2': w2, 'Cosine Similarity': round(float(sim_score), 4)})

df_sim = pd.DataFrame(similarity_records)
print(df_sim)

=== 5.7: Cosine Similarity Between Word Pairs ===
    Word 1    Word 2  Cosine Similarity
0  machine  learning            -0.1137
1  natural  language            -0.0099
2     deep  learning            -0.0442
